In [ ]:
import matplotlib.pyplot as plt          # For plotting and visualization
from sklearn.datasets import fetch_openml  # To load MNIST dataset
import numpy as np                       # For numerical operations
from scipy.sparse import coo_matrix      # For sparse matrix operations
from sklearn.neighbors import NearestNeighbors  # For finding nearest neighbors

# Load the MNIST dataset
print("Loading MNIST dataset...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
# fetch_openml: Downloads MNIST from OpenML repository
# 'mnist_784': 28x28 images flattened to 784 features
# version=1: Original MNIST version
# as_frame=False: Return numpy arrays

X = mnist.data                          # Feature matrix: shape (70000, 784)
                                        # Each row = one handwritten digit image
                                        # Each column = one pixel value (0-255)

y = mnist.target.astype(int)            # Labels: shape (70000,)
                                        # Convert from string to integer (0-9)

# Use a small subset for faster computation
n_samples = 1000                        # Use 1000 samples (UMAP is faster than t-SNE)
X_subset = X[:n_samples]                # Shape: (1000, 784)
y_subset = y[:n_samples]                # Shape: (1000,)

# Normalize the data
X_subset = X_subset / 255.0             # Scale pixel values to [0, 1]
                                        # Helps with distance calculations

print("Implementing UMAP from scratch...")

# Step 1: Find k-nearest neighbors
def compute_knn_graph(X, n_neighbors=15):
    """
    Compute k-nearest neighbors graph
    For each point, find its k nearest neighbors
    """
    print("Computing k-nearest neighbors...")
    
    # Use scikit-learn's NearestNeighbors for efficient computation
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    # n_neighbors: number of nearest neighbors to find
    # metric='euclidean': use Euclidean distance
    
    nn.fit(X)                           # Fit the model on data
    
    # Find k-nearest neighbors for each point
    distances, indices = nn.kneighbors(X)
    # distances: shape (n_samples, n_neighbors) - distances to neighbors
    # indices: shape (n_samples, n_neighbors) - indices of neighbors
    
    return distances, indices

# Step 2: Compute local connectivity (fuzzy simplicial set)
def compute_membership_strengths(distances, indices, local_connectivity=1.0):
    """
    Compute membership strengths for the fuzzy simplicial set
    This creates a weighted graph where edge weights represent probabilities
    """
    print("Computing membership strengths...")
    
    n_samples = distances.shape[0]      # Number of samples
    n_neighbors = distances.shape[1]    # Number of neighbors per sample
    
    # Initialize arrays for sparse matrix
    rows = np.zeros(n_samples * n_neighbors, dtype=np.int32)
    # rows: row indices for sparse matrix
    
    cols = np.zeros(n_samples * n_neighbors, dtype=np.int32)
    # cols: column indices for sparse matrix
    
    vals = np.zeros(n_samples * n_neighbors, dtype=np.float32)
    # vals: values (edge weights) for sparse matrix
    
    # Compute rho (distance to nearest neighbor) and sigma for each point
    rho = distances[:, 1]               # Distance to 1st nearest neighbor (skip self at index 0)
                                        # Shape: (n_samples,)
    
    # For each point, compute sigma (normalization factor)
    sigma = np.zeros(n_samples, dtype=np.float32)
    for i in range(n_samples):
        # Compute distances beyond the nearest neighbor
        non_zero_dists = distances[i, 1:] - rho[i]
        # Subtract rho to get relative distances
        
        # Estimate sigma using mean of non-zero distances
        non_zero_dists = non_zero_dists[non_zero_dists > 0]
        if len(non_zero_dists) > 0:
            sigma[i] = np.mean(non_zero_dists)
        else:
            sigma[i] = 1.0              # Default value if no valid distances
    
    # Compute membership strengths (edge weights)
    for i in range(n_samples):
        for j in range(n_neighbors):
            # For each neighbor of point i
            if distances[i, j] == 0:    # Skip self-loops (distance = 0)
                continue
            
            # Compute membership strength using exponential kernel
            # val = exp(-(dist - rho) / sigma)
            val = np.exp(-(distances[i, j] - rho[i]) / sigma[i])
            
            # Store in sparse matrix format
            idx = i * n_neighbors + j   # Index in flattened arrays
            rows[idx] = i               # Row index (source node)
            cols[idx] = indices[i, j]   # Column index (target node)
            vals[idx] = val             # Edge weight (membership strength)
    
    # Create sparse matrix
    graph = coo_matrix((vals, (rows, cols)), shape=(n_samples, n_samples))
    # COO (coordinate) format: stores (row, col, value) triplets
    
    # Symmetrize the graph: combine A and A^T
    graph = graph + graph.T             # Add transpose to make symmetric
    
    return graph.tocsr()                # Convert to CSR format for efficient operations

# Step 3: Initialize low-dimensional embedding
def initialize_embedding(n_samples, n_components=2):
    """
    Initialize the low-dimensional embedding randomly
    Start with small random values near the origin
    """
    print("Initializing embedding...")
    
    # Random initialization with small values
    embedding = np.random.randn(n_samples, n_components) * 0.01
    # Shape: (n_samples, 2)
    # Small values prevent large initial gradients
    
    return embedding

# Step 4: Optimize embedding using gradient descent
def optimize_embedding(embedding, graph, n_epochs=200, learning_rate=1.0, min_dist=0.1):
    """
    Optimize the low-dimensional embedding using stochastic gradient descent
    Minimizes the difference between high-dim and low-dim probability distributions
    """
    print("Optimizing embedding...")
    
    n_samples = embedding.shape[0]      # Number of samples
    
    # Get edges from the graph (non-zero elements)
    graph_coo = graph.tocoo()           # Convert to COO format
    edges = list(zip(graph_coo.row, graph_coo.col))  # List of (i, j) edges
    weights = graph_coo.data            # Edge weights
    
    # Parameters for optimization
    a = 1.0                             # Curve parameter for low-dim probability
    b = 1.0                             # Curve parameter for low-dim probability
    
    # Run optimization for n_epochs
    for epoch in range(n_epochs):
        # Shuffle edges for stochastic gradient descent
        indices = np.random.permutation(len(edges))
        
        # Process each edge
        for idx in indices:
            i, j = edges[idx]           # Get edge (i, j)
            weight = weights[idx]       # Get edge weight (high-dim probability)
            
            # Compute distance in low-dimensional space
            diff = embedding[i] - embedding[j]  # Difference vector
            dist_squared = np.sum(diff ** 2)    # Squared distance
            
            # Avoid division by zero
            if dist_squared < 1e-8:
                dist_squared = 1e-8
            
            # Compute attractive force (bring similar points closer)
            # Based on high-dimensional edge weight
            grad_coeff_attract = -2.0 * a * b * weight * (dist_squared ** (b - 1))
            grad_coeff_attract /= (a * (dist_squared ** b) + 1.0)
            
            # Compute repulsive force (push dissimilar points apart)
            # Based on negative sampling (implicit repulsion)
            grad_coeff_repel = 2.0 * b
            grad_coeff_repel /= ((dist_squared + 1e-8) * (a * (dist_squared ** b) + 1.0))
            
            # Combined gradient
            grad_coeff = grad_coeff_attract + grad_coeff_repel
            
            # Update embeddings using gradient descent
            gradient = grad_coeff * diff
            embedding[i] += learning_rate * gradient
            embedding[j] -= learning_rate * gradient
        
        # Print progress every 50 epochs
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch + 1}/{n_epochs}")
    
    return embedding

# Step 5: Run simplified UMAP algorithm
def umap_manual(X, n_components=2, n_neighbors=15, min_dist=0.1, n_epochs=200):
    """
    Simplified UMAP implementation
    Steps:
    1. Find k-nearest neighbors
    2. Compute fuzzy simplicial set (weighted graph)
    3. Initialize low-dimensional embedding
    4. Optimize embedding using gradient descent
    """
    n_samples = X.shape[0]
    
    # Step 1: Compute k-nearest neighbors
    distances, indices = compute_knn_graph(X, n_neighbors)
    
    # Step 2: Compute membership strengths (fuzzy simplicial set)
    graph = compute_membership_strengths(distances, indices)
    
    # Step 3: Initialize embedding
    embedding = initialize_embedding(n_samples, n_components)
    
    # Step 4: Optimize embedding
    embedding = optimize_embedding(embedding, graph, n_epochs, learning_rate=1.0, min_dist=min_dist)
    
    return embedding

# Run simplified UMAP
print("\nRunning simplified UMAP algorithm...")
Y_umap = umap_manual(X_subset, n_components=2, n_neighbors=15, min_dist=0.1, n_epochs=200)
# Input: (1000, 784) → Output: (1000, 2)

# Visualize the results
plt.figure(figsize=(12, 10))           # Create figure

scatter = plt.scatter(
    Y_umap[:, 0],                      # X-axis: first UMAP component
    Y_umap[:, 1],                      # Y-axis: second UMAP component
    c=y_subset,                        # Color by digit label (0-9)
    cmap='tab10',                      # 10 distinct colors
    alpha=0.7,                         # 70% opacity
    s=30                               # Point size
)

# Add labels and title
plt.xlabel('UMAP Component 1', fontsize=12)
plt.ylabel('UMAP Component 2', fontsize=12)
plt.title(f'Simplified UMAP Implementation on MNIST ({n_samples} samples)', 
          fontsize=14, fontweight='bold')

# Add colorbar
plt.colorbar(scatter, label='Digit Label')

# Add grid
plt.grid(True, alpha=0.3)

# Adjust layout and display
plt.tight_layout()
plt.show()

print("\nSimplified UMAP visualization complete!")

# Print statistics
print("\nUMAP Parameters:")
print(f"  - Input dimensions: {X_subset.shape[1]}")     # 784
print(f"  - Output dimensions: {Y_umap.shape[1]}")      # 2
print(f"  - Number of samples: {Y_umap.shape[0]}")      # 1000
print(f"  - Number of neighbors: 15")
print(f"  - Minimum distance: 0.1")
print(f"  - Number of epochs: 200")

Loading MNIST dataset...
Implementing UMAP from scratch...

Running simplified UMAP algorithm...
Computing k-nearest neighbors...
Computing membership strengths...
Initializing embedding...
Optimizing embedding...
